Section - 01: Import libraries

In [1]:
import pandas as pd
import numpy as np 

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width",1000)

In [4]:
appointments = pd.read_csv(
    "../data/appointments.csv"
)

clinics = pd.read_csv(
    "../data/clinics.csv"
)

print("Appointments:", appointments.shape)
print("Clinics:", clinics.shape)

Appointments: (30, 3)
Clinics: (8, 6)


In [10]:
# Reload the appointment-level data if appointments currently references another table
appointments = pd.read_csv("../data/appointments.csv")

required = {"clinic_id", "appointment_id", "no_show"}
missing = required - set(appointments.columns)
if missing:
    raise KeyError(f"appointments.csv is missing columns: {sorted(missing)}")

clinic_analysis = (
    appointments.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

clinic_analysis["no_show_pct"] = (
    clinic_analysis["no_shows"] /
    clinic_analysis["appointments"] * 100
).round(1)

clinic_analysis = clinic_analysis.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    how="left"
)

clinic_analysis = clinic_analysis[
    [
        "clinic_name",
        "city",
        "appointments",
        "no_shows",
        "no_show_pct"
    ]
].sort_values(
    "no_show_pct",
    ascending=False
)

display(clinic_analysis)

KeyError: "appointments.csv is missing columns: ['clinic_id', 'no_show']"

In [11]:
clinic_analysis.to_csv(
    "../data/validation/sql_python_comparison.csv",
    index=False
)

NameError: name 'clinic_analysis' is not defined

In [12]:
sql_results = pd.read_csv(
    "../data/validation/sql_clinic_results.csv"
)

comparison = clinic_analysis.merge(
    sql_results,
    on=["clinic_name", "city"],
    suffixes=("_python", "_sql")
)

comparison["counts_match"] = (
    (comparison["appointments_python"] ==
     comparison["appointments_sql"]) &
    (comparison["no_shows_python"] ==
     comparison["no_shows_sql"])
)

display(comparison)

FileNotFoundError: [Errno 2] No such file or directory: '../data/validation/sql_clinic_results.csv'